# 任务③：LLM 辅助的轨迹清洗评估工作流

本 notebook 演示一条完整闭环：

```
诊断 → 检索记忆 → LLM 提议 → 执行 → 核验 → 记忆
```

**核心设计**：LLM 只负责"选工具、提参数、说依据"；
所有数值计算在确定性代码里；**提议好不好由 regret 定量判定**，
而不是靠人去读一段解释。

只有通过核验的建议才允许写入记忆——这是整个工作流的准入闸门。

---
### 目录
1. 环境与数据
2. 单条轨迹完整闭环（逐步展示 trace）
3. 出图：清洗前后叠加 / 异常分布 / 热力图
4. 消融实验：llm-only / search-only / llm+search / llm+memory+search
5. 记忆的可解释视图
6. 切换到真实 LLM API


## 1. 环境与数据

In [ ]:
import os, sys, json, warnings
from pathlib import Path
from dotenv import load_dotenv

# 本地密钥保存在被 Git 忽略的 .env；Notebook 中不写入真实 key
load_dotenv(Path(".env"), override=False)
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("."))

# matplotlib 需要可写缓存目录
os.environ.setdefault("MPLCONFIGDIR", os.path.abspath(".mplcache"))

from traj_agent.core import traj as tm, diagnosis, params as params_mod
from traj_agent.tools.registry import ToolRegistry, attach_dataset
from traj_agent.agent.loop import TrajCleaningAgent
from traj_agent.agent import provider as prov
from traj_agent.memory.store import MemoryStore
from traj_agent.tools.playbook import vault_status

DATA = "traj_dict.json"
raw = tm.load_raw(DATA)
print(f"载入 {len(raw)} 辆车")

status = prov.provider_status()
print("\nLLM provider 配置：")
for k, v in status.items():
    print(f"  {k}: {v}")
print("\nObsidian vault：", json.dumps(vault_status(), ensure_ascii=False))


### 数据集的两个 regime

本项目约 63% 的车辆全程静止、37% 在真实行驶。
**对这两类用同一套阈值是不可能的** —— 这是智能体应当自动识别出的第一件事。

In [2]:
import collections
sample = sorted(raw.keys(), key=lambda k: (len(k), k))[:120]
cards = [diagnosis.diagnose(tm.traj_from_raw(v, *raw[v])) for v in sample]
summary = diagnosis.dataset_regime_summary(cards)
print(json.dumps(summary, ensure_ascii=False, indent=1))


{
 "n_cards": 120,
 "regimes": {
  "stationary": 104,
  "mixed": 15,
  "moving": 1
 },
 "timelines": {
  "ok": 116,
  "degraded": 4
 }
}


## 2. 单条轨迹完整闭环

In [3]:
mem = MemoryStore(":memory:")      # 演示用内存库；生产换成 sqlite 文件路径
agent = TrajCleaningAgent(llm=prov.MockProvider(), memory=mem,
                          regret_threshold=0.05).attach_dataset(raw)

VEHICLE = "246"     # 11 km 的真实行驶轨迹
result = agent.run(VEHICLE)
print(f"ok={result.ok}  来源={result.proposal_source}  "
      f"工具调用={result.tool_calls}  LLM 轮次={result.llm_turns}")


ok=True  来源=llm  工具调用=4  LLM 轮次=5


### 诊断卡（这就是 LLM 看到的全部数据 —— 不含任何坐标）

In [4]:
print(json.dumps(result.diagnosis, ensure_ascii=False, indent=1))

{
 "seg_id": "246#0",
 "vehicle_id": "246",
 "n_points": 108,
 "duration_s": 1190.0,
 "regime": "moving",
 "is_stationary": false,
 "timeline": {
  "quality": "ok",
  "dt_median_s": 10.0,
  "dt_p95_s": 20.0,
  "dt_max_s": 28.0,
  "dt_zero_ratio": 0.0093,
  "dt_zero_burst": 1,
  "dt_bimodal": false
 },
 "space": {
  "length_m": 11326.54,
  "displacement_m": 8377.85,
  "bbox_span_m": 8816.03,
  "sinuosity": 1.352
 },
 "duplicates": {
  "consecutive_dup_ratio": 0.0,
  "dup_run_count": 0
 },
 "speed": {
  "median_mps": 9.51,
  "p95_mps": 27.31,
  "max_mps": 253.18
 },
 "anomalies": {
  "counts": {
   "accel_outlier": 6,
   "drift": 1,
   "dt_artifact": 4,
   "nonpositive_dt": 1,
   "speed_over_limit": 3,
   "stationary_drift": 1
  },
  "n_anomalous_points": 9
 },
 "preserved_behavior": {
  "speed_over_limit": 3,
  "u_turn": 1
 },
 "road_match_rate": null,
 "observations": {
  "speed": "最大速度 253 m/s 明显超出城市物理上限：优先怀疑时间戳伪影而非坐标错误。"
 }
}


### trace：智能体每一步做了什么

这张表就是课堂讲评的材料 —— 能看到 LLM 调了哪些工具、
每次核验算出了什么数。

In [5]:
for s in result.trace:
    detail = json.dumps(s.detail, ensure_ascii=False)
    print(f"[{s.index:2d}] {s.kind:6s} {s.name:22s} {detail[:110]}")

[ 0] tool   load_trajectory        {"handle": "246#0@v0", "n_points": 108}
[ 1] memory retrieve               {"available": false, "n_neighbors": 0, "n_regions": 0}
[ 2] llm    direct                 {"content": "（Mock）调用 clean_trajectory", "n_tool_calls": 0}
[ 3] llm    direct_no_json         {"reason": "直答未含有效 JSON，升级到带工具路径"}
[ 4] llm    turn1                  {"content": "（Mock）调用 split_trajectory", "n_tool_calls": 1}
[ 5] tool   split_trajectory       {"args": {"handle": "246#0@v0", "dt_threshold": 30.0, "dist_threshold": 400.0}, "ok": true, "error": ""}
[ 6] llm    turn2                  {"content": "（Mock）调用 run_search", "n_tool_calls": 1}
[ 7] tool   run_search             {"args": {"handle": "246#0@v0", "param": "dp_tolerance", "n": 7, "reference_handle": "246#0@v0"}, "ok": true, 
[ 8] llm    turn3                  {"content": "（Mock）调用 find_knee", "n_tool_calls": 1}
[ 9] tool   find_knee              {"args": {"curve_json": "[{\"value\": 0.5, \"n_points\": 85, \"compression_ra

### 提议与核验结果

`regret` 是**可自动判分**的核心数字：
相对确定性搜索找到的最优解，这条建议丢掉了多少比例的可得收益。

In [6]:
d = result.to_dict(with_trace=False)
print("LLM 提议的参数：")
print(json.dumps(d["proposal_params"], ensure_ascii=False, indent=1))
print("\n它的依据：")
print(result.proposal_raw.get("rationale", ""))
print("\n它预测的效果（会被逐一比对符号）：")
print(json.dumps(result.proposal_raw.get("expected_effect", {}), ensure_ascii=False))
print("\n实测指标（对清洗后轨迹，隔离出纯 DP 误差）：")
print(json.dumps({k: v for k, v in d["measured"].items() if k != "clean_report"},
                 ensure_ascii=False, indent=1))


LLM 提议的参数：
{
 "dist_threshold": 400.0,
 "dp_tolerance": 15.25,
 "dt_threshold": 30.0,
 "max_speed_mps": 38.0
}

它的依据：
依据实测容差扫描的 knee point 取 dp_tolerance=15.25 m；该点之后压缩收益递减而偏差开始超过 GPS 精度。时间阈值沿用 30s（若发现 Δt 双峰应改用 p95）。

它预测的效果（会被逐一比对符号）：
{"quality": "down", "compression": "up", "dp_tolerance": "up"}

实测指标（对清洗后轨迹，隔离出纯 DP 误差）：
{
 "n_points_before": 106,
 "n_points_after": 19,
 "compression_ratio": 0.8208,
 "max_deviation_m": 11.9098,
 "length_before_m": 11322.087,
 "length_after_m": 11290.639,
 "hausdorff_m": 11.91,
 "frechet_m": 943.345,
 "runtime_ms": 0.7534
}


In [7]:
v = d["verification"]
print(f"提议得分   : {v['proposal_score']}")
print(f"基线得分   : {v['baseline_score']}   （物理先验默认参数）")
print(f"搜索最优   : {v['best_score']}")
print(f"regret     : {v['regret']}   口径={v['regret_basis']}")
print(f"方向准确率 : {v['direction_accuracy']}")
print(f"约束满足   : {v['constraint_ok']}   夹紧项={v['clamped_params']}")
print(f"记忆准入   : {v['admitted']}  —— {v['admit_reason']}")

提议得分   : 0.600277
基线得分   : 0.632211   （物理先验默认参数）
搜索最优   : 0.635887
regret     : 0.03561   口径=absolute(small_headroom)
方向准确率 : 1.0
约束满足   : True   夹紧项=[]
记忆准入   : False  —— regret 0.0356（absolute(small_headroom)）超过阈值 0.01


## 3. 出图

In [8]:
from traj_agent.core import clean, simplify
from traj_agent.report import figures

t_raw = tm.traj_from_raw(VEHICLE, *raw[VEHICLE])
t_clean, report = clean.denoise_trajectory(t_raw)
t_simp = simplify.simplify_trajectory(t_clean, d["proposal_params"]["dp_tolerance"]).traj

print("清洗账本：")
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=1))

p1 = figures.plot_clean_overlay(t_simp, reference=t_raw, out_dir="figures",
                                filename=f"notebook_overlay_{VEHICLE}.png")
p2 = figures.plot_anomaly_scatter(t_raw, out_dir="figures",
                                  filename=f"notebook_anomaly_{VEHICLE}.png")
p3 = figures.plot_heatmap(t_simp, reference=t_raw, out_dir="figures",
                          filename=f"notebook_heatmap_{VEHICLE}.png")
print("\n输出：", p1, p2, p3, sep="\n  ")


清洗账本：
{
 "n_before": 108,
 "n_after": 106,
 "dropped": 2,
 "drop_ratio": 0.0185,
 "actions": {
  "drop_jump_artifact": 1,
  "drop_nonpositive_dt": 1,
  "fix_dt_artifact": 9
 },
 "dropped_reasons": {
  "nonpositive_dt": 1,
  "space_jump": 1,
  "speed_over_limit": 1
 },
 "dropped_with_reason": 2,
 "preserved_behavior": {
  "speed_over_limit": 2,
  "u_turn": 1
 },
 "n_iterations": 3
}



输出：
  figures\notebook_overlay_246.png
  figures\notebook_anomaly_246.png
  figures\notebook_heatmap_246.png


### 质量—压缩率曲线

这张图**同时服务任务②和任务③**：
任务②看阈值扫描的边际递减形状，
任务③它就是 verifier 计算 regret 的目标函数地形图。

In [9]:
from traj_agent.core import segment

reg = ToolRegistry(); attach_dataset(reg.ctx, raw)
h = reg.call("load_trajectory", vehicle_id=VEHICLE)["handle"]
hc = reg.call("clean_trajectory", handle=h)["handle"]
curve = reg.call("run_search", handle=hc, param="dp_tolerance", n=8,
                 reference_handle=hc)["curve"]
knee = reg.call("find_knee", curve_json=json.dumps(curve))["knee"]
p4 = figures.plot_quality_compression(curve, knee, out_dir="figures",
                                     filename=f"notebook_curve_{VEHICLE}.png")

print(f"{'容差':>8} {'点数':>6} {'压缩率':>8} {'最大偏差':>10} {'长度比':>8}")
for c in curve:
    print(f"{c['value']:8.1f} {c['n_points']:6d} {c['compression_ratio']:8.3f} "
          f"{c['max_deviation_m']:10.2f} {c['length_ratio']:8.4f}")
print("\nknee point：", json.dumps(knee, ensure_ascii=False))


      容差     点数      压缩率       最大偏差      长度比
     0.5     83    0.217       0.34   1.0000
     4.7     36    0.660       4.43   0.9988
     8.9     26    0.755       8.33   0.9976
    13.1     19    0.821      11.91   0.9972
    17.4     18    0.830      15.31   0.9972
    21.6     18    0.830      15.31   0.9972
    25.8     18    0.830      15.31   0.9972
    30.0     16    0.849      29.20   0.9969

knee point： {"index": 2, "tolerance_m": 0.0, "n_points": 26, "compression": 0.7547, "fidelity": 0.7223, "method": "curvature", "value": 8.9286, "max_deviation_m": 8.331, "deviation_scale_m": 30.0, "max_score_alternative": {"value": 4.7143, "compression": 0.6604, "fidelity": 0.8522}}


In [10]:
rows = segment.sweep_split_thresholds(t_clean, [15, 30, 60, 120],
                                        [100, 200, 400, 800, 1600])
p5 = figures.plot_sensitivity_heatmap(rows, out_dir="figures",
                                     filename=f"notebook_sensitivity_{VEHICLE}.png")
p6 = figures.plot_ablation.__doc__ and None  # 占位，消融图在第 4 节出
print("敏感性热图：", p5)

敏感性热图： figures\notebook_sensitivity_246.png


## 4. 消融实验

四种模式必须**结构性不同**，否则消融表没有意义。

| 模式 | LLM | 记忆 | 搜索 |
|---|---|---|---|
| llm-only | 有 | 无 | 无 |
| search-only | **无** | 无 | 有 |
| llm+search | 有 | 无 | 有 |
| llm+memory+search | 有 | 有 | 有 |

### 两个必须避开的陷阱

**1. 信息泄漏。** 若在评测轨迹上边跑边攒记忆，agent 会把「这条轨迹自己的
上一次结果」检索回来当先验 —— 等于考试时把答案摆桌上。
故 `run_ablation` 强制两阶段：demo 集积累记忆，**留出集**只读不写（`read_only=True`），
且 `retrieve_similar(exclude_self=True)` 排除自身。
demo 与 holdout 重叠会直接抛错。

**2. regret 口径不可比。** 关闭搜索时本模式可达上限就是基线，
regret 退化为绝对口径；含搜索模式是归一化口径。
两者**不能**直接比较，代码会在 `caveats` 里标注。

下面用按 (regime, timeline) 分层挑选的 demo / holdout 两组（各 12 条，零重叠）。


In [11]:
from traj_agent.verifier.verify import ablation_summary

specs = [
    ("llm-only",          dict(use_llm=True,  use_memory=False, use_search=False, regret_threshold=1.0)),
    ("search-only",       dict(use_llm=False, use_memory=False, use_search=True,  regret_threshold=0.05)),
    ("llm+search",        dict(use_llm=True,  use_memory=False, use_search=True,  regret_threshold=0.05)),
    ("llm+memory+search", dict(use_llm=True,  use_memory=True,  use_search=True,  regret_threshold=1.0)),
]
VEHICLES = ["246", "256", "306", "209"]

cases, per_mode = [], {}
for mode, kw in specs:
    m = MemoryStore(":memory:") if kw["use_memory"] else None
    r0 = ToolRegistry(); attach_dataset(r0.ctx, raw)
    ag = TrajCleaningAgent(registry=r0, llm=prov.MockProvider(), memory=m,
                           mode=mode, **kw)
    per_mode[mode] = []
    for vid in VEHICLES:
        r = ag.run(vid)
        if not r.ok:
            continue
        dd = r.to_dict(with_trace=False); vv = dd["verification"]
        row = {"mode": mode, "regret": vv["regret"],
               "proposal_score": vv["proposal_score"],
               "n_evaluations": dd["tool_calls"],
               "direction_accuracy": vv["direction_accuracy"],
               "admitted": vv["admitted"]}
        cases.append(row); per_mode[mode].append(row)
    if m: m.close()
    print(f"{mode:22s} 完成 {len(per_mode[mode])} 条")

rows = ablation_summary(cases)
print()
print(f"{'模式':22s} {'n':>3} {'平均regret':>10} {'平均分':>8} {'平均调用':>8} {'方向准确':>8} {'准入率':>7}")
for r in rows:
    print(f"{r['mode']:22s} {r['n']:3d} {str(r['mean_regret']):>10} {str(r['mean_score']):>8} "
          f"{str(r['mean_evals']):>8} {str(r['mean_direction_accuracy']):>8} {str(r['admit_rate']):>7}")

p6 = figures.plot_ablation(rows, out_dir="figures", filename="notebook_ablation.png")
print("\n消融图：", p6)


llm-only               完成 4 条


search-only            完成 4 条


llm+search             完成 4 条


llm+memory+search      完成 4 条

模式                       n   平均regret      平均分     平均调用     方向准确     准入率
llm+memory+search        4       0.02   0.5701      4.0      1.0    0.25
llm+search               4       0.02     0.57      4.0      1.0    0.25
llm-only                 4     0.0153     0.57      4.0      1.0    0.25
search-only              4     0.0075   0.5826      1.0     None     1.0



消融图： figures\notebook_ablation.png


### 消融结果解读（离线演示）

本次使用 `MockProvider` 和 4 条指定车辆轨迹。`search-only` 的平均 regret 为 0.0075，准入率 100%；三个包含模拟 LLM 的模式平均 regret 约 0.0153–0.0200，准入率均为 25%。在这组小样本上，加入模拟提议没有改善核验结果。4 条轨迹不足以推断真实模型的整体效果，且 `MockProvider` 不是实际 LLM；应在留出集上扩大样本，并在有 API 凭据时单独验证真实模型。

## 5. 记忆的可解释视图

记忆检索用 **12 维归一化特征 + regime 硬门控**，不用 embedding。

这样做的理由：特征都有物理含义，学生能把检索到的邻居和特征值直接打出来看，
而不是面对一个黑盒向量。 `regime` 作为门控（而非一个距离维度）是因为
静止轨迹与行驶轨迹的特征分布完全不同，混在一起检索会让结果失去参考价值。

In [12]:
from traj_agent.memory import retrieve as retrieve_mod
from traj_agent.memory import features as feat_mod

card = diagnosis.diagnose(t_raw)
view = retrieve_mod.explain_neighbors(card, agent.memory, k=3)
print("查询轨迹的特征向量：")
print(json.dumps(view["query"]["features"], ensure_ascii=False, indent=1))
print("\n检索到的邻居：")
for nb in view["neighbors"]:
    print(f"  {nb['seg_id']:10s} 相似度={nb['similarity']:.4f} regime匹配={nb['regime_match']} "
          f"参数={nb['params']} regret={nb['regret']}")
print("\n说明：", view["note"])


查询轨迹的特征向量：
{
 "log_n_points": 0.8846,
 "duration_ratio": 0.9917,
 "log_dt_median": 0.5833,
 "dt_cv": 0.5,
 "dup_ratio": 0.0,
 "log_length": 0.9218,
 "log_displacement": 0.9121,
 "sinuosity": 0.1173,
 "log_speed_median": 0.722,
 "log_speed_max": 1.0,
 "anomaly_density": 0.0833,
 "dt_zero_ratio": 0.0093
}

检索到的邻居：

说明： 检索用 12 维归一化特征 + regime 硬门控，不用 embedding。特征都有物理含义，可直接归因。


In [13]:
# L2：从已准入案例蒸馏出的参数区间
n = agent.memory.rebuild_procedural(min_samples=1)
print(f"蒸馏出 {n} 条参数区间\n")
print(retrieve_mod.region_hint_text(retrieve_mod.prior_for(card, agent.memory)))
print("\n记忆库统计：")
print(json.dumps(agent.memory.stats(), ensure_ascii=False, indent=1))

蒸馏出 0 条参数区间

（无已验证的参数区间）

记忆库统计：
{
 "n_episodic": 1,
 "n_admitted": 0,
 "admit_rate": 0.0,
 "n_procedural": 0,
 "by_regime": [],
 "path": ":memory:"
}


### 时间轴故障的轨迹

车辆 352 的重复时间戳占比约 68.6%，最长连续 39 个点落在同一时刻。实测最大速度达到 242 m/s，优先怀疑时间戳故障。若直接把所有高速度点当作坐标漂移删除，可能损失有效位置；应先修复时间轴或隔离故障段，再评估空间清洗。

In [14]:
r352 = agent.run("352")
d352 = r352.to_dict(with_trace=False)
print("时间轴诊断：", json.dumps(r352.diagnosis["timeline"], ensure_ascii=False))
print("observations：", json.dumps(r352.diagnosis["observations"], ensure_ascii=False, indent=1))
print("\n提议：", json.dumps(d352["proposal_params"], ensure_ascii=False))
print("依据：", r352.proposal_raw.get("rationale", ""))
print(f"\nregret={d352['verification']['regret']:.4f} 准入={d352['verification']['admitted']}")
print("理由：", d352["verification"]["admit_reason"])

时间轴诊断： {"quality": "degraded", "dt_median_s": 10.0, "dt_p95_s": 20.0, "dt_max_s": 34.0, "dt_zero_ratio": 0.6864, "dt_zero_burst": 39, "dt_bimodal": false}
observations： {
 "timeline": "重复时间戳占比 68.6%，且最长连续 39 个点落在同一时刻（突发式）：这是采集端时间戳故障，删点会损失大量有效坐标，应先修时间戳或整段隔离。",
 "speed": "最大速度 242 m/s 明显超出城市物理上限：优先怀疑时间戳伪影而非坐标错误。"
}

提议： {"dist_threshold": 400.0, "dp_tolerance": 5.42, "dt_threshold": 30.0, "max_speed_mps": 38.0}
依据： 依据实测容差扫描的 knee point 取 dp_tolerance=5.4167 m；该点之后压缩收益递减而偏差开始超过 GPS 精度。时间阈值沿用 30s（若发现 Δt 双峰应改用 p95）。

regret=0.5343 准入=False
理由： regret 0.5343（normalized）超过阈值 0.05


### 静止轨迹：目标函数不适用

车辆 0 清洗后只有 21 点、总长 12.29 m，全是厘米级抖动。
此时任何容差都会让长度比崩掉 —— 但这不是"方案不好"，
而是**压缩率与保真度在这个尺度上没有物理含义**。

核验器对此显式标记 `applicable=False`，改按 regime 正确性准入，
而不是强行算一个负分或虚高的 regret。

In [15]:
r0 = agent.run("0")
d0 = r0.to_dict(with_trace=False)
print("regime:", d0["regime"], " 长度相关:", json.dumps(r0.diagnosis["space"], ensure_ascii=False))
print("proposal_objective:", json.dumps(d0["proposal_objective"], ensure_ascii=False, indent=1))
print("\n核验：", json.dumps(d0["verification"], ensure_ascii=False, indent=1))

regime: stationary  长度相关: {"length_m": 12.29, "displacement_m": 4.99, "bbox_span_m": 6.66, "sinuosity": 2.4611}
proposal_objective: {
 "n_points": 4,
 "n_points_ref": 21,
 "tolerance_m": 0.5,
 "max_deviation_m": 0.361,
 "length_ratio": 0.8555,
 "hausdorff_m": 0.0,
 "frechet_m": 0.0,
 "road_match_rate": null,
 "runtime_ms": 0.081,
 "compression": 0.0,
 "fidelity": 0.0,
 "road_term": 0.0,
 "runtime_term": 0.0,
 "score": 0.0,
 "feasible": true,
 "violations": [],
 "applicable": false,
 "inapplicable_reason": "轨迹总长 12.3 m 低于 200 m，压缩率与保真度在该尺度上没有物理含义"
}

核验： {
 "constraint_ok": true,
 "clamped_params": [],
 "feasible": true,
 "violations": [],
 "proposal_score": 0.0,
 "baseline_score": 0.0,
 "best_score": 0.0,
 "regret": 0.0,
 "regret_basis": "not_applicable",
 "applicable": false,
 "inapplicable_reason": "轨迹总长 12.3 m 低于 200 m，压缩率与保真度在该尺度上没有物理含义",
 "direction_checks": [
  {
   "param": "quality",
   "predicted": "same",
   "observed": "same",
   "agree": true
  },
  {
   "param": "compressi

## 6. 切换到真实 LLM API

Notebook 开头会读取当前目录中被 Git 忽略的 `.env`。ECNU 配置格式如下，真实 key 只写入本地 `.env`，不要直接写进 Notebook：

```dotenv
DSH_TRAJ_LLM_API_KEY=你的令牌
DSH_TRAJ_LLM_BASE_URL=https://chat.ecnu.edu.cn/open/api/v1
DSH_TRAJ_LLM_MODEL=ecnu-plus
```

`build_provider()` 会在检测到 key 后返回 `OpenAICompatProvider`，否则回退到 `MockProvider`。前面的课程演示仍显式使用 `MockProvider`，避免重新运行 Notebook 时意外消耗真实模型额度；需要执行真实模型实验时，应单独调用 `prov.build_provider()`，并把输出保存到新的实验目录。

In [ ]:
print(json.dumps(prov.provider_status(), ensure_ascii=False, indent=1))
print()
active_provider = prov.build_provider()
print("按当前配置将使用：", type(active_provider).__name__)
print("真实模型实验入口：TrajCleaningAgent(llm=prov.build_provider(), ...)")
print("离线演示入口：TrajCleaningAgent(llm=prov.MockProvider(), ...)")

## 小结：这个工作流解决了什么

| 问题 | 做法 |
|---|---|
| LLM 猜参数不准 | 物理先验定区间 → LLM 提起点 → 确定性搜索精调 |
| 无法判断 LLM 建议好不好 | regret = 相对搜索最优的差距，**可自动判分** |
| 每条轨迹都调 LLM 太贵 | 记忆分层：跑少数、复用多数（L2 直接给区间） |
| 记忆被幻觉污染 | **只有核验器能写记忆**；LLM 没有写记忆的工具 |
| 上下文塞不下坐标 | 坐标永不出进程，只以句柄 + 诊断卡流动 |
| 换模型要改代码 | OpenAI 兼容协议 + 文本 ReAct 兜底 |

### 下一步（不在本 notebook 范围）
- 路网约束接入真实 OSM 数据（`road/matcher.py` 已留协议）
- 批量跑全量 11386 条并做分层抽样统计
- 用真实 LLM 复跑消融表，对比 Mock 基线的差距
